# Brain Tumor Segmentation with U-Net Variants
**Dataset:** LGG MRI Segmentation (Kaggle – Mateusz Buda)  
**Task:** Segment FLAIR abnormality/tumor regions from brain MRI slices using manual masks.

This notebook is designed to run in **Google Colab** or locally. It:
1. Finds MRI/mask pairs and checks the dataset.
2. Splits by **patient**, not by slice, to reduce data leakage.
3. Trains a standard U-Net and an Attention U-Net.
4. Compares Dice, IoU, precision, recall, specificity and pixel accuracy.
5. Plots training curves and sample predictions, then saves the best models.

> Important: This is an educational segmentation experiment, not a diagnostic tool. Results depend on the dataset split, hardware, and training time.

## 1. Environment and imports

If TensorFlow is already installed, this cell will simply import it. In Colab, select **Runtime → Change runtime type → T4 GPU** when available.

In [ ]:
import os, glob, random, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

## 2. Get the dataset

Download and extract the Kaggle dataset from:
https://www.kaggle.com/datasets/mateuszbuda/lgg-mri-segmentation

**Colab option:** upload the extracted folder to Google Drive and mount Drive below.  
**Local option:** extract it and set `DATASET_ROOT` to the folder containing the patient folders.

Expected files are commonly arranged as patient folders containing MRI `.tif` images and corresponding files ending in `_mask.tif`. The discovery code below searches recursively, so the outer folder name can differ.

In [ ]:
# Google Colab: uncomment these two lines if your dataset is in Google Drive.
# from google.colab import drive
# drive.mount('/content/drive')

# EDIT THIS PATH to the extracted dataset folder:
DATASET_ROOT = "/content/lgg-mri-segmentation"

# Examples:
# DATASET_ROOT = "/content/drive/MyDrive/lgg-mri-segmentation"
# DATASET_ROOT = r"C:\\Users\\YourName\\Downloads\\lgg-mri-segmentation"

assert os.path.isdir(DATASET_ROOT), (
    f"Dataset folder not found: {DATASET_ROOT}\n"
    "Download/extract the Kaggle dataset and update DATASET_ROOT."
)

## 3. Discover and validate MRI-mask pairs

The mask filename is expected to be the MRI filename plus `_mask` before the extension. Each patient directory is treated as a group for splitting.

In [ ]:
root = Path(DATASET_ROOT)
mask_paths = sorted(root.rglob("*_mask.tif")) + sorted(root.rglob("*_mask.tiff"))
pairs = []
missing_images = []

for mask_path in mask_paths:
    name = mask_path.name
    if name.lower().endswith("_mask.tiff"):
        image_name = name[:-9] + ".tiff"
    else:
        image_name = name[:-9] + ".tif"  # remove "_mask.tif"
    image_path = mask_path.with_name(image_name)
    if image_path.exists():
        # Patient folder is the directory immediately containing the pair.
        patient_id = str(mask_path.parent.relative_to(root))
        pairs.append((str(image_path), str(mask_path), patient_id))
    else:
        missing_images.append(str(image_path))

# Remove accidental duplicate pairs while preserving order.
pairs = list(dict.fromkeys(pairs))
print(f"Found image-mask pairs: {len(pairs)}")
print(f"Masks without matching image: {len(missing_images)}")
print(f"Unique patient folders: {len(set(p[2] for p in pairs))}")
if len(pairs) == 0:
    print("A few files found under dataset root:")
    print([str(p.relative_to(root)) for p in list(root.rglob('*'))[:25]])
    raise RuntimeError("No pairs found. Check DATASET_ROOT and filename pattern.")

pairs[:3]

## 4. Patient-wise train/validation/test split

Slices from the same patient must not be distributed across train and test sets. We therefore split the patient IDs first, then assign all slices from each patient to one split.

In [ ]:
patient_ids = np.array(sorted(set(p[2] for p in pairs)))
train_patients, temp_patients = train_test_split(
    patient_ids, test_size=0.30, random_state=SEED
)
val_patients, test_patients = train_test_split(
    temp_patients, test_size=0.50, random_state=SEED
)

train_pairs = [p for p in pairs if p[2] in set(train_patients)]
val_pairs   = [p for p in pairs if p[2] in set(val_patients)]
test_pairs  = [p for p in pairs if p[2] in set(test_patients)]

print(f"Patients — train: {len(train_patients)}, val: {len(val_patients)}, test: {len(test_patients)}")
print(f"Slices   — train: {len(train_pairs)}, val: {len(val_pairs)}, test: {len(test_pairs)}")
assert not (set(train_patients) & set(val_patients))
assert not (set(train_patients) & set(test_patients))
assert not (set(val_patients) & set(test_patients))

## 5. Inspect images and masks

MRI images are converted to RGB and resized to a fixed square input size. Masks are converted to grayscale, resized with nearest-neighbor interpolation, and binarized.

In [ ]:
IMG_SIZE = 256
BATCH_SIZE = 16
AUTOTUNE = tf.data.AUTOTUNE

def load_pair_numpy(image_path, mask_path):
    image = Image.open(image_path).convert("RGB").resize(
        (IMG_SIZE, IMG_SIZE), Image.Resampling.BILINEAR
    )
    mask = Image.open(mask_path).convert("L").resize(
        (IMG_SIZE, IMG_SIZE), Image.Resampling.NEAREST
    )
    image = np.asarray(image, dtype=np.float32) / 255.0
    mask = (np.asarray(mask, dtype=np.uint8) > 0).astype(np.float32)
    return image, mask[..., np.newaxis]

fig, axes = plt.subplots(2, 3, figsize=(12, 7))
for ax, pair in zip(axes.flat, random.sample(pairs, min(6, len(pairs)))):
    img, msk = load_pair_numpy(pair[0], pair[1])
    ax.imshow(img)
    ax.imshow(msk[..., 0], cmap="Reds", alpha=0.35)
    ax.set_title(Path(pair[2]).name)
    ax.axis("off")
plt.suptitle("MRI slices with manual tumor masks (red overlay)")
plt.tight_layout()
plt.show()

## 6. Build TensorFlow input pipelines

Images and masks are loaded from disk, normalized, and batched. Light augmentation (flips and 90-degree rotations) is applied only to training data.

In [ ]:
def _load_tf(image_path, mask_path):
    image_bytes = tf.io.read_file(image_path)
    mask_bytes = tf.io.read_file(mask_path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    mask = tf.io.decode_image(mask_bytes, channels=1, expand_animations=False)
    image.set_shape([None, None, 3])
    mask.set_shape([None, None, 1])
    image = tf.image.resize(tf.cast(image, tf.float32) / 255.0,
                            [IMG_SIZE, IMG_SIZE], method="bilinear")
    mask = tf.image.resize(tf.cast(mask, tf.float32),
                           [IMG_SIZE, IMG_SIZE], method="nearest")
    mask = tf.cast(mask > 0, tf.float32)
    return image, mask

def _augment(image, mask):
    if tf.random.uniform(()) > 0.5:
        image, mask = tf.image.flip_left_right(image), tf.image.flip_left_right(mask)
    if tf.random.uniform(()) > 0.5:
        image, mask = tf.image.flip_up_down(image), tf.image.flip_up_down(mask)
    k = tf.random.uniform((), minval=0, maxval=4, dtype=tf.int32)
    image, mask = tf.image.rot90(image, k), tf.image.rot90(mask, k)
    return image, mask

def make_dataset(pair_list, training=False):
    image_paths = [p[0] for p in pair_list]
    mask_paths = [p[1] for p in pair_list]
    ds = tf.data.Dataset.from_tensor_slices((image_paths, mask_paths))
    if training:
        ds = ds.shuffle(len(pair_list), seed=SEED, reshuffle_each_iteration=True)
    ds = ds.map(_load_tf, num_parallel_calls=AUTOTUNE)
    if training:
        ds = ds.map(_augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_pairs, training=True)
val_ds = make_dataset(val_pairs)
test_ds = make_dataset(test_pairs)

## 7. Loss functions and evaluation metrics

Tumor pixels occupy a relatively small portion of many slices, so Dice loss is combined with binary cross-entropy. Metrics are calculated on the binarized predictions at a 0.5 threshold.

In [ ]:
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

def combined_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return tf.reduce_mean(bce) + dice_loss(y_true, y_pred)

def build_conv_block(x, filters):
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Activation("relu")(x)
    x = tf.keras.layers.Conv2D(filters, 3, padding="same", kernel_initializer="he_normal")(x)
    x = tf.keras.layers.BatchNormalization()(x)
    return tf.keras.layers.Activation("relu")(x)

def attention_gate(skip, gating, filters):
    theta = tf.keras.layers.Conv2D(filters, 1, padding="same")(skip)
    phi = tf.keras.layers.Conv2D(filters, 1, padding="same")(gating)
    add = tf.keras.layers.Activation("relu")(tf.keras.layers.Add()([theta, phi]))
    psi = tf.keras.layers.Conv2D(1, 1, padding="same")(add)
    alpha = tf.keras.layers.Activation("sigmoid")(psi)
    return tf.keras.layers.Multiply()([skip, alpha])

def build_unet(input_shape=(IMG_SIZE, IMG_SIZE, 3), attention=False):
    inputs = tf.keras.Input(input_shape)
    c1 = build_conv_block(inputs, 32); p1 = tf.keras.layers.MaxPooling2D()(c1)
    c2 = build_conv_block(p1, 64); p2 = tf.keras.layers.MaxPooling2D()(c2)
    c3 = build_conv_block(p2, 128); p3 = tf.keras.layers.MaxPooling2D()(c3)
    c4 = build_conv_block(p3, 256); p4 = tf.keras.layers.MaxPooling2D()(c4)
    bn = build_conv_block(p4, 512)

    u4 = tf.keras.layers.UpSampling2D(interpolation="bilinear")(bn)
    s4 = attention_gate(c4, u4, 256) if attention else c4
    u4 = tf.keras.layers.Concatenate()([u4, s4]); c5 = build_conv_block(u4, 256)

    u3 = tf.keras.layers.UpSampling2D(interpolation="bilinear")(c5)
    s3 = attention_gate(c3, u3, 128) if attention else c3
    u3 = tf.keras.layers.Concatenate()([u3, s3]); c6 = build_conv_block(u3, 128)

    u2 = tf.keras.layers.UpSampling2D(interpolation="bilinear")(c6)
    s2 = attention_gate(c2, u2, 64) if attention else c2
    u2 = tf.keras.layers.Concatenate()([u2, s2]); c7 = build_conv_block(u2, 64)

    u1 = tf.keras.layers.UpSampling2D(interpolation="bilinear")(c7)
    s1 = attention_gate(c1, u1, 32) if attention else c1
    u1 = tf.keras.layers.Concatenate()([u1, s1]); c8 = build_conv_block(u1, 32)

    outputs = tf.keras.layers.Conv2D(1, 1, activation="sigmoid")(c8)
    return tf.keras.Model(inputs, outputs,
                          name="Attention_UNet" if attention else "U_Net")

## 8. Train the standard U-Net

Training defaults are deliberately moderate for a Colab GPU. Increase epochs if time and GPU memory allow. Early stopping and learning-rate reduction help avoid unnecessary training.

In [ ]:
EPOCHS = 20  # Increase to 30–50 for a longer experiment if resources allow.
def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss=combined_loss,
        metrics=[dice_coef, tf.keras.metrics.BinaryIoU(target_class_ids=[1], threshold=0.5, name="tumor_iou")]
    )
    return model

def callbacks_for(name):
    return [
        tf.keras.callbacks.ModelCheckpoint(
            f"best_{name}.keras", monitor="val_dice_coef",
            mode="max", save_best_only=True, verbose=1
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor="val_dice_coef", mode="max", patience=6,
            restore_best_weights=True, verbose=1
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1
        )
    ]

unet = compile_model(build_unet(attention=False))
unet.summary()
start = time.time()
history_unet = unet.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=callbacks_for("unet")
)
print(f"U-Net training time: {(time.time()-start)/60:.1f} minutes")

## 9. Train Attention U-Net

Attention gates learn to emphasize relevant spatial regions in the encoder skip connections. The architecture is otherwise kept comparable to the standard U-Net.

In [ ]:
attention_unet = compile_model(build_unet(attention=True))
attention_unet.summary()
start = time.time()
history_attention = attention_unet.fit(
    train_ds, validation_data=val_ds, epochs=EPOCHS,
    callbacks=callbacks_for("attention_unet")
)
print(f"Attention U-Net training time: {(time.time()-start)/60:.1f} minutes")

## 10. Compare training curves

In [ ]:
def plot_histories(histories, metric):
    plt.figure(figsize=(8, 5))
    for label, hist in histories.items():
        if metric in hist.history:
            plt.plot(hist.history[metric], label=f"{label} train")
        if f"val_{metric}" in hist.history:
            plt.plot(hist.history[f"val_{metric}"], linestyle="--", label=f"{label} val")
    plt.xlabel("Epoch"); plt.ylabel(metric); plt.title(f"{metric} comparison")
    plt.legend(); plt.grid(alpha=0.25); plt.show()

plot_histories({"U-Net": history_unet, "Attention U-Net": history_attention}, "loss")
plot_histories({"U-Net": history_unet, "Attention U-Net": history_attention}, "dice_coef")
plot_histories({"U-Net": history_unet, "Attention U-Net": history_attention}, "tumor_iou")

## 11. Test-set evaluation

Dice and IoU are computed per image and averaged across test slices. Precision, recall, specificity and pixel accuracy are calculated from the pooled pixel-level confusion counts. These are **pixel-level** metrics, not patient-level diagnostic metrics.

In [ ]:
def evaluate_segmentation(model, dataset, threshold=0.5):
    tp = fp = tn = fn = 0
    dice_scores, iou_scores = [], []
    for images, masks in dataset:
        probs = model.predict(images, verbose=0)
        preds = (probs >= threshold).astype(np.uint8)
        true = masks.numpy().astype(np.uint8)
        for y, p in zip(true, preds):
            y = y.reshape(-1); p = p.reshape(-1)
            tpi = np.sum((y == 1) & (p == 1))
            fpi = np.sum((y == 0) & (p == 1))
            tni = np.sum((y == 0) & (p == 0))
            fni = np.sum((y == 1) & (p == 0))
            tp += tpi; fp += fpi; tn += tni; fn += fni
            dice_scores.append((2*tpi + 1e-7) / (2*tpi + fpi + fni + 1e-7))
            iou_scores.append((tpi + 1e-7) / (tpi + fpi + fni + 1e-7))
    precision = tp / (tp + fp + 1e-7)
    recall = tp / (tp + fn + 1e-7)
    specificity = tn / (tn + fp + 1e-7)
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-7)
    return {
        "Dice": float(np.mean(dice_scores)),
        "IoU": float(np.mean(iou_scores)),
        "Precision": float(precision),
        "Recall / Sensitivity": float(recall),
        "Specificity": float(specificity),
        "Pixel Accuracy": float(accuracy)
    }

results = {}
for name, model in [("U-Net", unet), ("Attention U-Net", attention_unet)]:
    results[name] = evaluate_segmentation(model, test_ds)
results_df = pd.DataFrame(results).T
display(results_df.round(4))

## 12. Visualize predictions on unseen test slices

In [ ]:
def show_predictions(model, pair_list, n=5, threshold=0.5):
    samples = random.sample(pair_list, min(n, len(pair_list)))
    fig, axes = plt.subplots(len(samples), 4, figsize=(13, 3.2*len(samples)))
    if len(samples) == 1:
        axes = np.expand_dims(axes, 0)
    for row, (img_path, mask_path, patient) in enumerate(samples):
        image, mask = load_pair_numpy(img_path, mask_path)
        pred = model.predict(image[np.newaxis, ...], verbose=0)[0, ..., 0]
        axes[row, 0].imshow(image); axes[row, 0].set_title("MRI")
        axes[row, 1].imshow(mask[..., 0], cmap="gray"); axes[row, 1].set_title("Ground truth")
        axes[row, 2].imshow(pred, cmap="viridis", vmin=0, vmax=1); axes[row, 2].set_title("Predicted probability")
        axes[row, 3].imshow(pred >= threshold, cmap="gray"); axes[row, 3].set_title("Predicted mask")
        for ax in axes[row]: ax.axis("off")
    plt.tight_layout(); plt.show()

show_predictions(unet, test_pairs)
show_predictions(attention_unet, test_pairs)

## 13. Save models and results

The best checkpoint files are saved during training. This cell also saves the final models and a CSV comparison table.

In [ ]:
unet.save("unet_final.keras")
attention_unet.save("attention_unet_final.keras")
results_df.to_csv("segmentation_test_results.csv")
print("Saved:")
print("- best_unet.keras / best_attention_unet.keras (best validation checkpoints)")
print("- unet_final.keras / attention_unet_final.keras (final trained models)")
print("- segmentation_test_results.csv")

## 14. Final result summary

Use the table produced in Section 11 for your actual reported scores. Do not copy example or expected metrics: segmentation scores vary with the patient-wise split, training duration, and hardware.

**Suggested conclusion:** The experiment compares a baseline U-Net with an Attention U-Net for pixel-wise segmentation of FLAIR abnormalities in brain MRI. Dice and IoU measure mask overlap, while precision and recall describe false-positive and false-negative tumor pixels. Visual inspection is important because a single aggregate score may hide poor segmentation on individual slices.

### Limitations
- This dataset contains FLAIR abnormality masks for a low-grade glioma cohort; it is not a general brain-tumor dataset.
- A slice-wise score does not establish patient-level diagnostic performance.
- Patient-wise splitting reduces leakage, but performance still depends on the small number of patients.
- Predictions should not be used for clinical decisions.